# 05 SequencerWidget Tutorial

Program step patterns by length, bars, and rhythmic resolution.

## Grid Length and Resolution

A sequencer created with `length=8` represents one bar of eighth notes by default, so the visual grid, step selector, and bar summary all agree.

In [ ]:
from nbplay import SequencerWidget

one_bar_eighths = SequencerWidget(length=8, bpm=120.0)
for step, note in enumerate([60, 64, 67, 72, 67, 64, 60, 55]):
    one_bar_eighths.set_step(step, note=note, velocity=100, active=True)

one_bar_eighths

In [ ]:
print(f'measures: {one_bar_eighths.measures}')
print(f'step_duration: {one_bar_eighths.step_duration}')
print(f'length: {one_bar_eighths.length}')
assert one_bar_eighths.step_duration == 0.5
assert one_bar_eighths.length == 8

## Configure Bars and Step Size

Use `configure_grid()` to resize a pattern musically. Existing notes are preserved while new steps are added with defaults.

In [ ]:
four_bar_sixteenths = SequencerWidget(num_voices=2, bpm=124.0)
four_bar_sixteenths.configure_grid(measures=4, step_duration=0.25)

for step in range(0, four_bar_sixteenths.length, 4):
    four_bar_sixteenths.set_step(step, note=48, velocity=112, active=True, voice=0)

for step, note in zip(range(2, four_bar_sixteenths.length, 8), [60, 62, 64, 67, 69, 67, 64, 62]):
    four_bar_sixteenths.set_step(step, note=note, velocity=90, active=True, voice=1)

four_bar_sixteenths

In [ ]:
print(f'voices: {four_bar_sixteenths.num_voices}')
print(f'length: {four_bar_sixteenths.length}')
print(f'voice lengths: {[len(voice) for voice in four_bar_sixteenths.voices_data]}')
assert four_bar_sixteenths.measures == 4
assert four_bar_sixteenths.length == 64

## Transport Time Signature Sync

When a sequencer is added to a `Session`, transport BPM and time signature stay linked to the sequencer grid configuration.

In [ ]:
from nbplay import Session, SynthWidget

session = Session(bpm=128.0, time_signature=(3, 4))
transport_seq = SequencerWidget()
transport_seq.configure_grid(measures=1, step_duration=0.5)
track = session.add_track('Lead', transport_seq, SynthWidget())

print(session.transport.time_signature_num, session.transport.time_signature_den)
print(transport_seq.time_signature_num, transport_seq.time_signature_den)
print(transport_seq.length)
assert transport_seq.time_signature_num == 3
assert transport_seq.time_signature_den == 4
assert transport_seq.length == 6
track

## NoteComposer and Pattern Export

Each sequencer voice is a `NoteComposer`. Convert a voice to a Rust `Pattern` when you want the lower-level representation.

In [ ]:
composer = four_bar_sixteenths.composers[0]
pattern = composer.to_pattern(loop_enabled=four_bar_sixteenths.loop_enabled)
print(composer)
print(pattern)
assert len(pattern) == four_bar_sixteenths.length